In [4]:
import os
from glob import glob
import pandas as pd

PASTA_BASE = r"C:\Industria E\IndustriaE\Business Intelligence - Documentos\Relatório de Vendas de Produtos por Grupo"
PASTA_SAIDA = os.path.join(PASTA_BASE, "RELATORIOS_GERADOS")

os.makedirs(PASTA_SAIDA, exist_ok=True)

# Índices fixos da planilha-base
COL_GRUPO = 0       # A
COL_CATEGORIA = 1   # B
COL_CODIGO = 2      # C
COL_PRODUTO = 3     # D
COL_QTD = 4         # E
COL_TOTAL = 5       # F
COL_PERCENT = 6     # G


def ler_excel_sem_cabecalho(caminho):
    ext = os.path.splitext(caminho)[1].lower()

    if ext == ".xls":
        return pd.read_excel(caminho, header=None, dtype=object, engine="xlrd")
    return pd.read_excel(caminho, header=None, dtype=object)


def normalizar_mes(nome_arquivo):
    return os.path.splitext(os.path.basename(nome_arquivo))[0].strip()


def ordenar_meses(meses):
    meses = [str(m).strip() for m in meses]
    return sorted(meses)


def converter_numero_br(valor):
    if pd.isna(valor):
        return 0.0

    if isinstance(valor, (int, float)):
        return float(valor)

    texto = str(valor).strip()

    if texto == "":
        return 0.0

    texto = texto.replace(" ", "")
    texto = texto.replace(".", "").replace(",", ".")

    try:
        return float(texto)
    except ValueError:
        return 0.0


def tratar_planilha(caminho):
    df = ler_excel_sem_cabecalho(caminho)

    if df.shape[1] < 6:
        raise ValueError(f"Planilha com poucas colunas: {caminho}")

    # Mantém apenas A:G
    df = df.iloc[:, :7].copy()

    # Remove linhas totalmente vazias
    df = df.dropna(how="all").reset_index(drop=True)

    # Preenche células mescladas de grupo e categoria
    df[COL_GRUPO] = df[COL_GRUPO].ffill()
    df[COL_CATEGORIA] = df[COL_CATEGORIA].ffill()

    def linha_irrelevante(row):
        textos = [str(x).strip().upper() for x in row.tolist() if pd.notna(x)]
        if not textos:
            return True

        joined = " | ".join(textos)

        # Remove linha CONSOLIDADO
        if "CONSOLIDADO" in joined:
            return True

        # Remove linha com QUANTIDADE / TOTAL / %
        if "QUANTIDADE" in joined and "TOTAL" in joined:
            return True

        # Remove linha do nome do restaurante
        if len(textos) == 1 and "%" not in joined and "TOTAL" not in joined and "QUANTIDADE" not in joined:
            return True

        return False

    df = df[~df.apply(linha_irrelevante, axis=1)].copy()

    # Remove linhas de subtotal
    mask_total = (
        df[COL_CATEGORIA].astype(str).str.contains("TOTAL", case=False, na=False) |
        df[COL_PRODUTO].astype(str).str.contains("TOTAL", case=False, na=False)
    )
    df = df[~mask_total].copy()

    # Mantém somente linhas com código e produto
    df = df[
        df[COL_CODIGO].notna() &
        df[COL_PRODUTO].notna()
    ].copy()

    # Seleciona colunas finais
    df = df[[COL_PRODUTO, COL_CATEGORIA, COL_CODIGO, COL_QTD, COL_TOTAL]].copy()
    df.columns = ["PRODUTO", "CATEGORIA", "CODIGO", "QUANTIDADE", "TOTAL"]

    # Limpeza
    for col in ["PRODUTO", "CATEGORIA", "CODIGO"]:
        df[col] = df[col].astype(str).str.strip()

    # Mantém zeros à esquerda do código
    df["CODIGO"] = (
        df["CODIGO"]
        .astype(str)
        .str.strip()
        .str.replace(".0", "", regex=False)
        .str.zfill(5)
    )

    # Conversão numérica correta
    df["QUANTIDADE"] = df["QUANTIDADE"].apply(converter_numero_br)
    df["TOTAL"] = df["TOTAL"].apply(converter_numero_br)

    return df


def consolidar(df_total):
    pivot_qtd = df_total.pivot_table(
        index=["PRODUTO", "CATEGORIA", "CODIGO"],
        columns="MES",
        values="QUANTIDADE",
        aggfunc="sum",
        fill_value=0
    )

    pivot_total = df_total.pivot_table(
        index=["PRODUTO", "CATEGORIA", "CODIGO"],
        columns="MES",
        values="TOTAL",
        aggfunc="sum",
        fill_value=0
    )

    meses = ordenar_meses(list(set(pivot_qtd.columns).union(set(pivot_total.columns))))

    for mes in meses:
        if mes not in pivot_qtd.columns:
            pivot_qtd[mes] = 0
        if mes not in pivot_total.columns:
            pivot_total[mes] = 0

    pivot_qtd = pivot_qtd[meses]
    pivot_total = pivot_total[meses]

    base = pivot_qtd.reset_index()[["PRODUTO", "CATEGORIA", "CODIGO"]].copy()

    for mes in meses:
        base[(mes, "QUANTIDADE")] = pivot_qtd[mes].values
        base[(mes, "TOTAL")] = pivot_total[mes].values

    return base, meses


def adicionar_total_geral(df_final, meses):
    linha_total = {
        "PRODUTO": "TOTAL GERAL",
        "CATEGORIA": "",
        "CODIGO": ""
    }

    for mes in meses:
        linha_total[(mes, "QUANTIDADE")] = df_final[(mes, "QUANTIDADE")].sum()
        linha_total[(mes, "TOTAL")] = df_final[(mes, "TOTAL")].sum()

    return pd.concat([df_final, pd.DataFrame([linha_total])], ignore_index=True)


def exportar_excel(df_final, meses, restaurante):
    caminho_saida = os.path.join(PASTA_SAIDA, f"{restaurante}.xlsx")

    with pd.ExcelWriter(caminho_saida, engine="xlsxwriter") as writer:
        workbook = writer.book

        fmt_header = workbook.add_format({
            "bold": True,
            "bg_color": "#D9D9D9",
            "border": 1,
            "align": "center",
            "valign": "vcenter"
        })

        fmt_text = workbook.add_format({"border": 1})
        fmt_num_int = workbook.add_format({"border": 1, "num_format": "#,##0.00"})
        fmt_total_num = workbook.add_format({
            "bold": True,
            "bg_color": "#E2F0D9",
            "border": 1,
            "num_format": "#,##0.00"
        })
        fmt_total_text = workbook.add_format({
            "bold": True,
            "bg_color": "#E2F0D9",
            "border": 1
        })

        # =========================
        # ABA DADOS
        # =========================
        ws_dados = workbook.add_worksheet("DADOS")
        writer.sheets["DADOS"] = ws_dados

        # Linha 1: título e meses
        ws_dados.merge_range("A1:C1", restaurante, fmt_header)

        col_excel = 3
        for mes in meses:
            ws_dados.merge_range(0, col_excel, 0, col_excel + 1, mes, fmt_header)
            col_excel += 2

        # Linha 2: subcabeçalhos
        ws_dados.write(1, 0, "PRODUTO", fmt_header)
        ws_dados.write(1, 1, "CATEGORIA", fmt_header)
        ws_dados.write(1, 2, "CÓDIGO", fmt_header)

        col_excel = 3
        for _mes in meses:
            ws_dados.write(1, col_excel, "QUANTIDADE", fmt_header)
            ws_dados.write(1, col_excel + 1, "TOTAL", fmt_header)
            col_excel += 2

        # Dados
        linha_excel = 2
        for _, row in df_final.iterrows():
            eh_total = str(row["PRODUTO"]).strip().upper() == "TOTAL GERAL"

            ws_dados.write(linha_excel, 0, row["PRODUTO"], fmt_total_text if eh_total else fmt_text)
            ws_dados.write(linha_excel, 1, row["CATEGORIA"], fmt_total_text if eh_total else fmt_text)
            ws_dados.write(linha_excel, 2, row["CODIGO"], fmt_total_text if eh_total else fmt_text)

            col_excel = 3
            for mes in meses:
                qtd = float(row[(mes, "QUANTIDADE")])
                total = float(row[(mes, "TOTAL")])

                ws_dados.write_number(linha_excel, col_excel, qtd, fmt_total_num if eh_total else fmt_num_int)
                ws_dados.write_number(linha_excel, col_excel + 1, total, fmt_total_num if eh_total else fmt_num_int)

                col_excel += 2

            linha_excel += 1

        # Ajuste de largura
        ws_dados.set_column("A:A", 35)
        ws_dados.set_column("B:B", 28)
        ws_dados.set_column("C:C", 12)
        ws_dados.set_column(3, 3 + (len(meses) * 2), 14)

        # =========================
        # ABA DASHBOARD
        # =========================
        ws_dash = workbook.add_worksheet("DASHBOARD")
        writer.sheets["DASHBOARD"] = ws_dash

        ws_dash.write(0, 0, "MÊS", fmt_header)
        ws_dash.write(0, 1, "QUANTIDADE", fmt_header)
        ws_dash.write(0, 2, "FATURAMENTO", fmt_header)

        # ignora a última linha (TOTAL GERAL) ao montar o dashboard
        base_sem_total = df_final.iloc[:-1].copy()

        for i, mes in enumerate(meses, start=1):
            qtd_mes = float(base_sem_total[(mes, "QUANTIDADE")].sum())
            total_mes = float(base_sem_total[(mes, "TOTAL")].sum())

            ws_dash.write(i, 0, mes, fmt_text)
            ws_dash.write_number(i, 1, qtd_mes, fmt_num_int)
            ws_dash.write_number(i, 2, total_mes, fmt_num_int)

        grafico_fat = workbook.add_chart({"type": "column"})
        grafico_fat.add_series({
            "name": "Faturamento",
            "categories": ["DASHBOARD", 1, 0, len(meses), 0],
            "values": ["DASHBOARD", 1, 2, len(meses), 2],
        })
        grafico_fat.set_title({"name": "Faturamento por mês"})
        grafico_fat.set_size({"width": 720, "height": 320})
        ws_dash.insert_chart("E2", grafico_fat)

        grafico_qtd = workbook.add_chart({"type": "line"})
        grafico_qtd.add_series({
            "name": "Quantidade",
            "categories": ["DASHBOARD", 1, 0, len(meses), 0],
            "values": ["DASHBOARD", 1, 1, len(meses), 1],
        })
        grafico_qtd.set_title({"name": "Quantidade por mês"})
        grafico_qtd.set_size({"width": 720, "height": 320})
        ws_dash.insert_chart("E20", grafico_qtd)

        ws_dash.set_column("A:C", 18)


def processar_restaurante(pasta_restaurante):
    nome_restaurante = os.path.basename(pasta_restaurante)

    arquivos = glob(os.path.join(pasta_restaurante, "*.xls")) + glob(os.path.join(pasta_restaurante, "*.xlsx"))

    if not arquivos:
        print(f"Nenhum arquivo encontrado em: {nome_restaurante}")
        return

    lista = []

    for arquivo in arquivos:
        try:
            mes = normalizar_mes(arquivo)
            df = tratar_planilha(arquivo)
            df["MES"] = mes
            lista.append(df)
            print(f"OK: {nome_restaurante} -> {os.path.basename(arquivo)}")
        except Exception as e:
            print(f"ERRO em {arquivo}: {e}")

    if not lista:
        print(f"Nenhuma planilha válida processada para: {nome_restaurante}")
        return

    df_total = pd.concat(lista, ignore_index=True)

    df_final, meses = consolidar(df_total)
    df_final = adicionar_total_geral(df_final, meses)

    exportar_excel(df_final, meses, nome_restaurante)
    print(f"Relatório gerado: {nome_restaurante}")


def main():
    for nome in os.listdir(PASTA_BASE):
        pasta = os.path.join(PASTA_BASE, nome)

        if os.path.isdir(pasta) and nome.upper() != "RELATORIOS_GERADOS":
            processar_restaurante(pasta)

    print("\nFinalizado.")


if __name__ == "__main__":
    main()

OK: SELVAGEM -> 2026-01.xls
OK: SELVAGEM -> 2026-02.xls
OK: SELVAGEM -> 2026-03.xls
Relatório gerado: SELVAGEM
OK: VISTA BALCÃO -> 2026-01.xls
OK: VISTA BALCÃO -> 2026-02.xls
OK: VISTA BALCÃO -> 2026-03.xls
Relatório gerado: VISTA BALCÃO
OK: VISTA CAFÉ -> 2026-01.xls
OK: VISTA CAFÉ -> 2026-02.xls
OK: VISTA CAFÉ -> 2026-03.xls
Relatório gerado: VISTA CAFÉ
OK: VISTA IBIRAPUERA -> 2026-01.xls
OK: VISTA IBIRAPUERA -> 2026-02.xls
OK: VISTA IBIRAPUERA -> 2026-03.xls
Relatório gerado: VISTA IBIRAPUERA

Finalizado.
